# 08 — Evaluación final sobre el conjunto de test

Este notebook realiza la evaluación final de **SciBERT Plus** sobre el conjunto de test reservado.

El modelo y su configuración fueron seleccionados exclusivamente mediante los resultados de validación. En esta etapa no se realiza entrenamiento, ajuste de hiperparámetros ni selección adicional del modelo.

La evaluación incluye:

- Accuracy y Macro F1.
- Precision, recall y F1 por clase.
- Matriz de confusión.
- Registro de resultados y artefactos en MLflow.
- Asociación de la evaluación con `CiteScope-SciBERT-Plus`, versión 1.

> El conjunto de test se evalúa una sola vez para conservar la validez de la estimación final.

## 1. Configuración y verificación de MLflow

Se establece la conexión con el servidor remoto de MLflow y se identifica la versión del modelo asociada al alias `candidate`. También se comprueba que la versión seleccionada todavía no haya sido marcada como evaluada sobre test.

In [1]:
import os
import mlflow
from mlflow import MlflowClient

# Configuración de MLflow
MLFLOW_TRACKING_URI = os.getenv(
    "MLFLOW_TRACKING_URI",
    "http://3.237.175.121:5000",
)

EXPERIMENT_NAME = "CiteScope - SciBERT Plus"
REGISTERED_MODEL_NAME = "CiteScope-SciBERT-Plus"
MODEL_ALIAS = "candidate"

mlflow.set_tracking_uri(MLFLOW_TRACKING_URI)
client = MlflowClient()

# Comprobar conexión y localizar el experimento
experiment = mlflow.get_experiment_by_name(EXPERIMENT_NAME)

if experiment is None:
    raise RuntimeError(
        f"No se encontró el experimento: {EXPERIMENT_NAME}"
    )

# Obtener la versión fijada como candidata
candidate_version = client.get_model_version_by_alias(
    REGISTERED_MODEL_NAME,
    MODEL_ALIAS,
)

version_tags = dict(candidate_version.tags)
test_evaluated = (
    version_tags.get("test_evaluated", "false").lower() == "true"
)

print("Tracking URI:", mlflow.get_tracking_uri())
print("Experimento:", experiment.name)
print("Experiment ID:", experiment.experiment_id)
print("Modelo registrado:", REGISTERED_MODEL_NAME)
print("Alias:", MODEL_ALIAS)
print("Versión:", candidate_version.version)
print("Run de origen:", candidate_version.run_id)
print("Test evaluado previamente:", test_evaluated)

if test_evaluated:
    raise RuntimeError(
        "Esta versión ya figura como evaluada sobre test. "
        "No se continuará para evitar una evaluación duplicada."
    )

print("\nVerificación completada: se puede preparar la evaluación final.")

d:\Documents\4. Courses\1. Master AI\1. Semesters\4 Semester\1 Bimestre\Grado_Microproyecto\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Tracking URI: http://3.237.175.121:5000
Experimento: CiteScope - SciBERT Plus
Experiment ID: 2
Modelo registrado: CiteScope-SciBERT-Plus
Alias: candidate
Versión: 1
Run de origen: 21250d1982d14958a93c64321b8b42b9
Test evaluado previamente: False

Verificación completada: se puede preparar la evaluación final.


## 2. Carga reproducible del conjunto de test

Se reconstruye la misma asignación de datos utilizada durante el entrenamiento mediante `split_assignment.csv`. Únicamente se conserva el subconjunto marcado como `test`, sin modificar su contenido ni utilizarlo para ajustar el modelo.

In [2]:
from pathlib import Path

import numpy as np
import pandas as pd
import torch

TARGET = "citing_primary_category"

EXPECTED_LABELS = [
    "cs.AI",
    "cs.CL",
    "cs.CV",
    "cs.IR",
    "cs.LG",
    "cs.MA",
    "cs.NE",
    "cs.RO",
]


def find_repo_root(start: Path) -> Path:
    for candidate in [start.resolve(), *start.resolve().parents]:
        if (
            (candidate / "Dataset").is_dir()
            and (candidate / "models").is_dir()
        ):
            return candidate

    raise FileNotFoundError(
        "No se encontró la raíz del repositorio."
    )


REPO_ROOT = find_repo_root(Path.cwd())
DATASET_DIR = REPO_ROOT / "Dataset"
ARTIFACTS_DIR = REPO_ROOT / "models" / "artifacts"

CANONICAL = DATASET_DIR / "unarxive_microproyecto.jsonl"
LOCAL_COPY = DATASET_DIR / "copy_unarxive_microproyecto.jsonl"
DATA_PATH = CANONICAL if CANONICAL.exists() else LOCAL_COPY
SPLIT_PATH = ARTIFACTS_DIR / "split_assignment.csv"

if not DATA_PATH.exists():
    raise FileNotFoundError(
        "No se encontró el dataset. Ejecuta `dvc pull`."
    )

if not SPLIT_PATH.exists():
    raise FileNotFoundError(
        "No se encontró models/artifacts/split_assignment.csv."
    )

# Cargar los datos y recuperar la asignación original de splits
df = pd.read_json(
    DATA_PATH,
    lines=True,
    dtype={"citing_arxiv_id": "string"},
)

split_map = pd.read_csv(SPLIT_PATH)[["citation_id", "split"]]

df = df.merge(
    split_map,
    on="citation_id",
    how="left",
    validate="one_to_one",
)

assert df["split"].notna().all(), "Existen registros sin split."
assert set(df["split"]) == {"train", "val", "test"}

# Reconstruir el mismo orden de clases empleado en el entrenamiento
labels = sorted(df[TARGET].unique().tolist())
assert labels == EXPECTED_LABELS, (
    f"El orden de clases no coincide: {labels}"
)

label2id = {
    label: index
    for index, label in enumerate(labels)
}
id2label = {
    index: label
    for label, index in label2id.items()
}

df["label"] = df[TARGET].map(label2id)

# Separar test sin mezclar ni reordenar observaciones
test = (
    df[df["split"] == "test"]
    .copy()
    .reset_index(drop=True)
)

assert len(test) == 800, (
    f"Se esperaban 800 observaciones de test y se encontraron {len(test)}."
)

assert test["label"].notna().all()

print("Dataset:", DATA_PATH.name)
print("Archivo de splits:", SPLIT_PATH.name)
print("Observaciones de test:", len(test))
print("Número de clases:", len(labels))
print("Orden de clases:", labels)

Dataset: unarxive_microproyecto.jsonl
Archivo de splits: split_assignment.csv
Observaciones de test: 800
Número de clases: 8
Orden de clases: ['cs.AI', 'cs.CL', 'cs.CV', 'cs.IR', 'cs.LG', 'cs.MA', 'cs.NE', 'cs.RO']


## 3. Carga del modelo y preparación de la entrada estructurada

Se carga el checkpoint seleccionado mediante validación y registrado como la versión 1 de `CiteScope-SciBERT-Plus`. Cada observación se tokeniza conservando presupuestos independientes para el contexto de citación, el título citado y el resumen citado.

Esta etapa solamente prepara el modelo y los datos; todavía no ejecuta la evaluación.

In [3]:
from transformers import (
    AutoModelForSequenceClassification,
    AutoTokenizer,
    DataCollatorWithPadding,
)

MODEL_CHECKPOINT = (
    ARTIFACTS_DIR
    / "scibert07_best_lr_1e-05_seed_42_ckpt"
    / "checkpoint-900"
)

MAX_LEN = 512
CTX_BUDGET = 192
TITLE_BUDGET = 48
ABSTRACT_BUDGET = 268

assert MODEL_CHECKPOINT.exists(), (
    f"No se encontró el checkpoint: {MODEL_CHECKPOINT}"
)

assert (
    1
    + CTX_BUDGET
    + 1
    + TITLE_BUDGET
    + 1
    + ABSTRACT_BUDGET
    + 1
    == MAX_LEN
)

device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

tokenizer = AutoTokenizer.from_pretrained(MODEL_CHECKPOINT)

model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_CHECKPOINT
)

model.to(device)
model.eval()


def field_tokens(value, budget):
    text = "" if pd.isna(value) else str(value).strip()

    return tokenizer.encode(
        text,
        add_special_tokens=False,
        truncation=True,
        max_length=budget,
    )


def encode_row(row):
    context_ids = field_tokens(
        row["citation_context"],
        CTX_BUDGET,
    )
    title_ids = field_tokens(
        row["cited_title"],
        TITLE_BUDGET,
    )
    abstract_ids = field_tokens(
        row["cited_abstract"],
        ABSTRACT_BUDGET,
    )

    input_ids = (
        [tokenizer.cls_token_id]
        + context_ids
        + [tokenizer.sep_token_id]
        + title_ids
        + [tokenizer.sep_token_id]
        + abstract_ids
        + [tokenizer.sep_token_id]
    )

    # Segmento 0: contexto de citación
    # Segmento 1: título y abstract del artículo citado
    first_segment = 1 + len(context_ids) + 1

    token_type_ids = (
        [0] * first_segment
        + [1] * (len(input_ids) - first_segment)
    )

    return {
        "input_ids": input_ids,
        "attention_mask": [1] * len(input_ids),
        "token_type_ids": token_type_ids,
        "labels": int(row["label"]),
    }


class StructuredDataset(torch.utils.data.Dataset):
    def __init__(self, frame):
        self.items = [
            encode_row(row)
            for _, row in frame.iterrows()
        ]

    def __len__(self):
        return len(self.items)

    def __getitem__(self, index):
        return self.items[index]


test_ds = StructuredDataset(test)

collator = DataCollatorWithPadding(
    tokenizer=tokenizer,
    return_tensors="pt",
)

lengths = np.array([
    len(item["input_ids"])
    for item in test_ds.items
])

print("Checkpoint:", MODEL_CHECKPOINT)
print("Dispositivo:", device)
print("Modelo en modo entrenamiento:", model.training)
print("Observaciones preparadas:", len(test_ds))
print("Longitud media:", round(lengths.mean(), 1))
print("Longitud p95:", int(np.percentile(lengths, 95)))
print("Longitud máxima:", lengths.max())

Checkpoint: D:\Documents\4. Courses\1. Master AI\1. Semesters\4 Semester\1 Bimestre\Grado_Microproyecto\models\artifacts\scibert07_best_lr_1e-05_seed_42_ckpt\checkpoint-900
Dispositivo: cuda
Modelo en modo entrenamiento: False
Observaciones preparadas: 800
Longitud media: 321.4
Longitud p95: 456
Longitud máxima: 486


## 4. Inferencia y métricas finales de test

Se ejecuta una única pasada de inferencia sobre las 800 observaciones reservadas. El modelo permanece en modo evaluación y se desactiva el cálculo de gradientes, por lo que sus parámetros no se modifican.

Las métricas principales son Accuracy y Macro F1. También se calcula el reporte por clase y la matriz de confusión para el análisis de errores.

In [4]:
import time

from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    f1_score,
)
from torch.utils.data import DataLoader
from tqdm.auto import tqdm

if globals().get("TEST_INFERENCE_COMPLETED", False):
    raise RuntimeError(
        "La inferencia de test ya fue ejecutada en esta sesión."
    )

BATCH_SIZE = 8

test_loader = DataLoader(
    test_ds,
    batch_size=BATCH_SIZE,
    shuffle=False,
    collate_fn=collator,
    pin_memory=torch.cuda.is_available(),
)

all_predictions = []
all_labels = []
all_probabilities = []

model.eval()
start_time = time.perf_counter()

with torch.inference_mode():
    for batch in tqdm(
        test_loader,
        desc="Evaluación de test",
    ):
        batch_labels = batch.pop("labels")

        inputs = {
            key: value.to(device, non_blocking=True)
            for key, value in batch.items()
        }

        outputs = model(**inputs)
        probabilities = torch.softmax(
            outputs.logits,
            dim=-1,
        )
        predictions = probabilities.argmax(dim=-1)

        all_labels.extend(
            batch_labels.cpu().numpy().tolist()
        )
        all_predictions.extend(
            predictions.cpu().numpy().tolist()
        )
        all_probabilities.append(
            probabilities.cpu().numpy()
        )

test_duration_seconds = time.perf_counter() - start_time

y_true = np.asarray(all_labels)
y_pred = np.asarray(all_predictions)
y_prob = np.concatenate(all_probabilities, axis=0)

assert len(y_true) == len(test) == 800
assert y_prob.shape == (800, len(labels))

test_accuracy = accuracy_score(y_true, y_pred)
test_macro_f1 = f1_score(
    y_true,
    y_pred,
    average="macro",
)
test_weighted_f1 = f1_score(
    y_true,
    y_pred,
    average="weighted",
)

test_report = classification_report(
    y_true,
    y_pred,
    labels=list(range(len(labels))),
    target_names=labels,
    output_dict=True,
    zero_division=0,
)

test_report_df = (
    pd.DataFrame(test_report)
    .transpose()
)

test_confusion_matrix = confusion_matrix(
    y_true,
    y_pred,
    labels=list(range(len(labels))),
)

TEST_INFERENCE_COMPLETED = True

print("\nEvaluación final completada")
print(f"Accuracy:    {test_accuracy:.6f}")
print(f"Macro F1:    {test_macro_f1:.6f}")
print(f"Weighted F1: {test_weighted_f1:.6f}")
print(f"Duración:    {test_duration_seconds:.1f} segundos")
print("\nReporte por clase:")
display(test_report_df.round(4))

Evaluación de test: 100%|██████████| 100/100 [00:10<00:00,  9.78it/s]


Evaluación final completada
Accuracy:    0.675000
Macro F1:    0.671616
Weighted F1: 0.671616
Duración:    10.2 segundos

Reporte por clase:


,precision,recall,f1-score,support
cs.AI,0.5441,0.370,0.4405,100.000
cs.CL,0.7358,0.780,0.7573,100.000
cs.CV,0.6058,0.830,0.7004,100.000
cs.IR,0.7889,0.710,0.7474,100.000
cs.LG,0.4732,0.530,0.5000,100.000
cs.MA,0.8608,0.680,0.7598,100.000
cs.NE,0.7010,0.680,0.6904,100.000
cs.RO,0.7387,0.820,0.7773,100.000
accuracy,0.6750,0.675,0.6750,0.675
macro avg,0.6811,0.675,0.6716,800.000


## 5. Registro de la evaluación final en MLflow

Se registran las métricas finales, el reporte por clase, la matriz de confusión y las predicciones obtenidas sobre test. La evaluación queda vinculada con la versión 1 del modelo registrado.

Después de completar correctamente el registro, la versión se marca con `test_evaluated=true`.

In [5]:
import json
import tempfile

import matplotlib.pyplot as plt
import seaborn as sns

if not globals().get("TEST_INFERENCE_COMPLETED", False):
    raise RuntimeError(
        "Primero debe completarse la inferencia de test."
    )

MODEL_VERSION = str(candidate_version.version)
EVALUATION_KEY = (
    f"{REGISTERED_MODEL_NAME}:v{MODEL_VERSION}:test"
)

# Evitar registrar accidentalmente la misma evaluación dos veces
existing_runs = mlflow.search_runs(
    experiment_ids=[experiment.experiment_id],
    filter_string=(
        f"tags.evaluation_key = '{EVALUATION_KEY}'"
    ),
    max_results=1,
)

if not existing_runs.empty:
    raise RuntimeError(
        "Esta evaluación de test ya está registrada en MLflow."
    )

predictions_df = pd.DataFrame({
    "citation_id": test["citation_id"],
    "true_id": y_true,
    "true_label": [id2label[int(i)] for i in y_true],
    "predicted_id": y_pred,
    "predicted_label": [id2label[int(i)] for i in y_pred],
    "confidence": y_prob.max(axis=1),
    "correct": y_true == y_pred,
})

for class_index, class_name in id2label.items():
    predictions_df[f"probability_{class_name}"] = (
        y_prob[:, class_index]
    )

summary = {
    "registered_model": REGISTERED_MODEL_NAME,
    "model_version": MODEL_VERSION,
    "model_alias_at_evaluation": MODEL_ALIAS,
    "source_run_id": candidate_version.run_id,
    "split": "test",
    "number_of_observations": int(len(test)),
    "number_of_classes": int(len(labels)),
    "accuracy": float(test_accuracy),
    "macro_f1": float(test_macro_f1),
    "weighted_f1": float(test_weighted_f1),
    "duration_seconds": float(test_duration_seconds),
    "labels": labels,
    "token_budgets": {
        "context": CTX_BUDGET,
        "title": TITLE_BUDGET,
        "abstract": ABSTRACT_BUDGET,
        "max_length": MAX_LEN,
    },
}

with tempfile.TemporaryDirectory() as temporary_directory:
    output_dir = Path(temporary_directory)

    report_path = output_dir / "classification_report_test.csv"
    predictions_path = output_dir / "predictions_test.csv"
    summary_path = output_dir / "test_evaluation_summary.json"
    confusion_path = output_dir / "confusion_matrix_test.png"

    test_report_df.to_csv(report_path, index=True)
    predictions_df.to_csv(predictions_path, index=False)

    summary_path.write_text(
        json.dumps(summary, indent=2, ensure_ascii=False),
        encoding="utf-8",
    )

    fig, ax = plt.subplots(figsize=(10, 8))

    sns.heatmap(
        test_confusion_matrix,
        annot=True,
        fmt="d",
        cmap="Blues",
        xticklabels=labels,
        yticklabels=labels,
        ax=ax,
    )

    ax.set_title("Matriz de confusión — Test")
    ax.set_xlabel("Clase predicha")
    ax.set_ylabel("Clase real")
    fig.tight_layout()
    fig.savefig(confusion_path, dpi=160)
    plt.close(fig)

    with mlflow.start_run(
        experiment_id=experiment.experiment_id,
        run_name="scibert_plus_test_evaluation_v1",
        tags={
            "evaluation_key": EVALUATION_KEY,
            "evaluation_stage": "final_test",
            "dataset_split": "test",
            "registered_model": REGISTERED_MODEL_NAME,
            "model_version": MODEL_VERSION,
            "model_alias": MODEL_ALIAS,
            "source_model_run_id": candidate_version.run_id,
            "test_evaluated": "true",
        },
    ) as evaluation_run:
        mlflow.log_params({
            "number_of_observations": len(test),
            "number_of_classes": len(labels),
            "batch_size": BATCH_SIZE,
            "max_length": MAX_LEN,
            "context_budget": CTX_BUDGET,
            "title_budget": TITLE_BUDGET,
            "abstract_budget": ABSTRACT_BUDGET,
            "checkpoint": MODEL_CHECKPOINT.name,
        })

        mlflow.log_metrics({
            "test_accuracy": test_accuracy,
            "test_macro_f1": test_macro_f1,
            "test_weighted_f1": test_weighted_f1,
            "test_duration_seconds": test_duration_seconds,
        })

        mlflow.log_artifacts(
            str(output_dir),
            artifact_path="test_evaluation",
        )

        evaluation_run_id = evaluation_run.info.run_id

# Marcar la versión solo después de registrar correctamente el run
model_version_tags = {
    "test_evaluated": "true",
    "test_accuracy": f"{test_accuracy:.6f}",
    "test_macro_f1": f"{test_macro_f1:.6f}",
    "test_weighted_f1": f"{test_weighted_f1:.6f}",
    "test_evaluation_run_id": evaluation_run_id,
}

for key, value in model_version_tags.items():
    client.set_model_version_tag(
        name=REGISTERED_MODEL_NAME,
        version=MODEL_VERSION,
        key=key,
        value=value,
    )

print("Evaluación registrada correctamente")
print("Run ID:", evaluation_run_id)
print("Modelo:", REGISTERED_MODEL_NAME)
print("Versión:", MODEL_VERSION)
print(f"Test Macro F1: {test_macro_f1:.6f}")
print("test_evaluated: true")

🏃 View run scibert_plus_test_evaluation_v1 at: http://3.237.175.121:5000/#/experiments/2/runs/93dd8040946243e4b6ca6f7645179335
🧪 View experiment at: http://3.237.175.121:5000/#/experiments/2
Evaluación registrada correctamente
Run ID: 93dd8040946243e4b6ca6f7645179335
Modelo: CiteScope-SciBERT-Plus
Versión: 1
Test Macro F1: 0.671616
test_evaluated: true


## 6. Promoción de la versión evaluada

Después de completar y registrar satisfactoriamente la evaluación final, la versión 1 se identifica con el alias `champion`. Este alias señala la versión aprobada para consumo por la API y no modifica los pesos ni los archivos del modelo.

In [6]:
CHAMPION_ALIAS = "champion"

client.set_registered_model_alias(
    name=REGISTERED_MODEL_NAME,
    alias=CHAMPION_ALIAS,
    version=MODEL_VERSION,
)

champion_version = client.get_model_version_by_alias(
    REGISTERED_MODEL_NAME,
    CHAMPION_ALIAS,
)

assert str(champion_version.version) == MODEL_VERSION

print("Promoción completada")
print("Modelo:", REGISTERED_MODEL_NAME)
print("Versión:", champion_version.version)
print("Alias asignado:", CHAMPION_ALIAS)
print(
    "URI para consumo:",
    f"models:/{REGISTERED_MODEL_NAME}@{CHAMPION_ALIAS}",
)

Promoción completada
Modelo: CiteScope-SciBERT-Plus
Versión: 1
Alias asignado: champion
URI para consumo: models:/CiteScope-SciBERT-Plus@champion
